# Baseline Model Preparation and Architecture Validation

Phase 8 • Controlled from-scratch comparison • Grayscale CE-MRI

This notebook prepares five standard torchvision baselines and validates their interfaces, classification gradients, complexity, storage, and batch-size memory. It does not train a model or evaluate held-out images.

## Public-copy execution policy
This is an English rendering of a historical research notebook. Outputs and execution counts were cleared; no cell was rerun for Phase 18. Historical findings below describe earlier runs only. Every code cell is gated, including setup cells that may perform model checks, preprocessing or writes. Enable `CONFIRM_PUBLIC_WORKFLOW` deliberately in the first code cell only after reviewing [the reproduction guide](../docs/reproducibility.md). Existing stage-specific confirmations must also be enabled separately. Run from the repository root or set `MORPHONET_ROOT` to a private writable experiment workspace with the same source/config layout; set `MORPHONET_DATA_ROOT` for dataset discovery. Original source/configuration fingerprints, private split manifests and checkpoints are not portable artifacts; do not bypass their validation. See the guide for prerequisites and limitations.


## Purpose and fair comparison protocol

The baselines are ResNet18, ResNet50, DenseNet121, MobileNetV3-Small, and EfficientNet-B0. Each receives the same 1 × 256 × 256 preprocessed MRI and predicts Meningioma (0), Glioma (1), or Pituitary (2). Only the first convolution and output class count are adapted; all deeper architecture settings are preserved.

All models explicitly use `weights=None`. Pretrained weights are rejected by the factory. Any future pretrained study must be a separate experiment. The proposed model is represented only by its saved Phase 5 complexity measurements; it is not rebuilt or loaded.

## Imports and reproducibility

The fixed seed is 42. Synthetic tensors are used for architecture, memory, and latency tests. Deterministic algorithms are enabled; cuDNN benchmarking and TF32 are disabled. Results include installed Python, PyTorch, torchvision, CUDA, and GPU details.

In [ ]:
CONFIRM_PUBLIC_WORKFLOW = False  # Explicit opt-in; no cell has been rerun.
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
from pathlib import Path
import sys
import json
import pandas as pd
import torch
import torchvision
import yaml
from IPython.display import display, Markdown

ROOT = Path(__import__("os").environ.get("MORPHONET_ROOT", str(Path.cwd()))).resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.models.baselines import BASELINES, DISPLAY_NAMES, INPUT_LAYERS, OUTPUT_LAYERS, build_baseline
from src.utils.baseline_setup import seed_everything, verify_data_protocol, run_model, save_setup, verify_protected, write_json

seed_everything()
OUTPUT = ROOT / "results/baselines/setup"
OUTPUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 10)
print(f"PyTorch {torch.__version__}; torchvision {torchvision.__version__}")
print("GPU:", torch.cuda.get_device_name(0))

## Frozen patient folds and preprocessing

All 15 split CSV files and the approved preprocessing are preserved. No patient assignments or class labels are regenerated. A hash snapshot protects existing checkpoints, training logs, Phase 7 results, architecture sources, and Phase 5 measurements.

**Approved augmentation policy:** training preprocessing may use the tumor mask internally to reject transforms that clip the tumor, exactly as in the frozen main experiment. The baseline dataset exposes only MRI, class label, PID, and sample ID. Auxiliary arrays produced internally by the frozen pipeline are discarded; they never enter the baseline model or loss. Validation and test request no targets and remain deterministic.

One training sample at a fixed epoch is compared bit-for-bit with the original image pipeline. No test images are accessed.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
data_audit, split_audit = verify_data_protocol(ROOT)
display(split_audit)
print("Training-image equivalence: PASS; exposed keys:", ", ".join(data_audit["dataset_keys"]))
print(f"Protected existing files: {data_audit['protected_files']}")

## Baseline factory and necessary adaptations

`build_baseline(model_name, num_classes=3, input_channels=1, pretrained=False)` accepts exactly the five canonical names below. Native constructors create three-class classifiers. First convolutions retain output channels, kernel, stride, padding, dilation, groups, and bias settings. The new grayscale kernels use the corresponding torchvision family initialization: Kaiming fan-in for DenseNet and fan-out for the other models. No RGB replication is applied.

Validation compares module topology, non-input settings, and all deeper initialized parameters against an identically seeded native three-class model. Analytical input-parameter reductions must match the actual counts.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
adaptations = pd.DataFrame({"Model": [DISPLAY_NAMES[name] for name in BASELINES],
                            "Input convolution": [INPUT_LAYERS[name] for name in BASELINES],
                            "Output classifier": [OUTPUT_LAYERS[name] for name in BASELINES]})
display(adaptations)
for arguments in [{"model_name": "unsupported"}, {"model_name": "resnet18", "pretrained": True},
                  {"model_name": "resnet18", "input_channels": 3}]:
    try:
        build_baseline(**arguments)
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid factory request was not rejected")
print("Unsupported-model, pretrained-weight, and input-policy guards: PASS.")

## Validation and measurement methods

Each disposable model completes FP32 forward and cross-entropy backward for two synthetic images. Logits must be finite with shape [2, 3]; all trainable parameter tensors must receive finite gradients, and the first and final parameter tensors must have nonzero gradient norms. Parameter hashes must remain unchanged. Training-mode BatchNorm buffers may update inside disposable smoke-test instances, which are never saved or reused for future training.

**Complexity:** executed Conv2d/Linear MACs at [1, 1, 256, 256], including the group divisor for depthwise layers. FLOPs = 2 × MACs. Bias additions, normalization, activations, pooling, residual adds, and gating multiplications are excluded. Independent forward hooks must equal THOP with all other registered operation rules zeroed. THOP default mixed counts are not used, and no extended-operation estimate is claimed.

**Memory:** batch 32 with CUDA FP16 autocast, FP32 parameters, classification loss, and backward. No optimizer or weight update is performed. Peak allocated/reserved memory is reported, along with a separate estimate for two AdamW moment tensors. Optimizer-step temporary allocations and gradient scaling are not measured; this is not a full-training memory guarantee. CUDA OOM is recorded without reducing batch size.

**Latency:** FP32, batch 1, evaluation mode, 30 warm-up iterations and 100 timed iterations with CUDA synchronization. Measurements are hardware-specific and do not alter weights.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
model_results = {}

## ResNet18: architecture and safety checks

Construct a fresh, randomly initialized model; validate the grayscale interface and classification gradients; measure Conv/Linear complexity, inference latency, and batch-32 memory. Only numerical measurements are retained.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
model_results["resnet18"] = run_model(ROOT, "resnet18")
display(pd.DataFrame([model_results["resnet18"]["summary"]])[
    ["Model", "Parameters", "TrainableParameters", "ConvLinearMACs", "ConvLinearFLOPs", "FP32MB"]])

## ResNet50: architecture and safety checks

Construct a fresh, randomly initialized model; validate the grayscale interface and classification gradients; measure Conv/Linear complexity, inference latency, and batch-32 memory. Only numerical measurements are retained.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
model_results["resnet50"] = run_model(ROOT, "resnet50")
display(pd.DataFrame([model_results["resnet50"]["summary"]])[
    ["Model", "Parameters", "TrainableParameters", "ConvLinearMACs", "ConvLinearFLOPs", "FP32MB"]])

## DenseNet121: architecture and safety checks

Construct a fresh, randomly initialized model; validate the grayscale interface and classification gradients; measure Conv/Linear complexity, inference latency, and batch-32 memory. Only numerical measurements are retained.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
model_results["densenet121"] = run_model(ROOT, "densenet121")
display(pd.DataFrame([model_results["densenet121"]["summary"]])[
    ["Model", "Parameters", "TrainableParameters", "ConvLinearMACs", "ConvLinearFLOPs", "FP32MB"]])

## MobileNetV3-Small: architecture and safety checks

Construct a fresh, randomly initialized model; validate the grayscale interface and classification gradients; measure Conv/Linear complexity, inference latency, and batch-32 memory. Only numerical measurements are retained.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
model_results["mobilenet_v3_small"] = run_model(ROOT, "mobilenet_v3_small")
display(pd.DataFrame([model_results["mobilenet_v3_small"]["summary"]])[
    ["Model", "Parameters", "TrainableParameters", "ConvLinearMACs", "ConvLinearFLOPs", "FP32MB"]])

## EfficientNet-B0: architecture and safety checks

Construct a fresh, randomly initialized model; validate the grayscale interface and classification gradients; measure Conv/Linear complexity, inference latency, and batch-32 memory. Only numerical measurements are retained.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
model_results["efficientnet_b0"] = run_model(ROOT, "efficientnet_b0")
display(pd.DataFrame([model_results["efficientnet_b0"]["summary"]])[
    ["Model", "Parameters", "TrainableParameters", "ConvLinearMACs", "ConvLinearFLOPs", "FP32MB"]])

## Parameter counts and frozen reference comparison

The proposed reference row is read from the Phase 5 JSON without constructing MorphoNet-BT. Counts include all parameters, while MACs describe the executed inference path. Baselines retain their standard deeper architectures. These measurements characterize resource cost, not predictive accuracy.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
summary, memory, latency = save_setup(ROOT, model_results, data_audit)
display(summary[["Model", "Role", "InputChannels", "Classes", "Pretrained", "Parameters", "TrainableParameters", "ParametersM", "TrainableParametersM"]])

## Computational complexity and FP32 parameter storage

Storage is parameter count × 4 bytes, with decimal MB (10⁶ bytes) and binary MiB (2²⁰ bytes). It excludes buffers, optimizer states, activations, and serialization overhead. All rows use exactly the same Conv/Linear MAC/FLOP convention; smaller parameter count need not imply lower latency.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
display(summary[["Model", "ConvLinearMACs", "ConvLinearFLOPs", "FP32Bytes", "FP32MB", "FP32MiB"]])
comparison = summary[["Model", "Parameters", "ConvLinearMACs"]].copy()
comparison["Parameter ratio to proposed"] = comparison.Parameters / 579599
comparison["MAC ratio to proposed"] = comparison.ConvLinearMACs / 275715968
display(comparison[["Model", "Parameter ratio to proposed", "MAC ratio to proposed"]])

## GPU memory safety at batch size 32

Every model is tested at the requested batch size. A PASS verifies this forward/loss/backward workload only. The optimizer moment estimate is additional persistent storage, not a measured training peak. Phase 9 must keep batch size 32 unless an explicit fairness decision is made.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
display(memory[["Model", "BatchSize", "Status", "PeakAllocatedMiB", "PeakReservedMiB", "AdamWTwoMomentStatesMiBEstimate"]])
print("No optimizer states or optimizer-step temporary allocations were included in the measured peaks.")

## Optional inference latency

These fresh measurements use the same GPU and the same general 30/100 synchronized batch-one protocol as Phase 5. They are not a contemporaneous rerun of the proposed model, so small latency differences should not be overinterpreted.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
display(latency[["Model", "MeanMs", "SDMs", "MedianMs", "ImagesPerSecond"]])

## Phase 9 configuration preparation

`configs/baselines.yaml` prepares 100 epochs, batch size 32, AdamW (LR 0.001, weight decay 0.0001), CosineAnnealingLR (T_max 100, eta_min 0), and early stopping on validation macro-F1 (patience 15, min_delta 0). CUDA AMP is enabled.

The sole objective is unweighted CrossEntropyLoss; no auxiliary loss is defined. Training remains disabled and requires separate Phase 9 approval. There is no baseline trainer invocation or saved baseline checkpoint in this notebook.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
configuration = yaml.safe_load((ROOT / "configs/baselines.yaml").read_text())
training = configuration["training"]
assert training["enabled"] is False and training["requires_phase9_approval"] is True
settings = {"Epoch limit": training["max_epochs"], "Batch size": training["batch_size"],
            "Optimizer": training["optimizer"]["name"], "Learning rate": training["optimizer"]["learning_rate"],
            "Weight decay": training["optimizer"]["weight_decay"], "Scheduler T_max": training["scheduler"]["T_max"],
            "Early-stop patience": training["early_stopping"]["patience"], "Loss": training["loss"]["name"],
            "Class weights": "Disabled", "Full training": "Not started"}
display(pd.DataFrame(settings.items(), columns=["Setting", "Value"]))

## Reproducibility metadata and final audit

Setup metadata records versions, device, seed, pretraining policy, measurement definitions, data checks, and source hashes. The final protected-file check verifies that existing main-experiment outputs and checkpoints remain byte-for-byte unchanged. The execution runner separately checks English-only notebook source/text outputs and successful cell execution.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
checks = {
    "Exactly five supported baselines": set(model_results) == set(BASELINES),
    "All FP32 forward/backward checks pass": all(r["validation"]["status"] == "PASS" for r in model_results.values()),
    "All input channels are one": all(r["summary"]["InputChannels"] == 1 for r in model_results.values()),
    "All classifiers return three logits": all(r["validation"]["logits_shape"] == [2, 3] for r in model_results.values()),
    "All weights are initialized from scratch": all(r["adaptation"]["weights"] is None for r in model_results.values()),
    "No optimizer steps": all(r["validation"]["optimizer_steps"] == 0 for r in model_results.values()),
    "Batch size 32 checked for all models": len(memory) == 5 and bool(memory.BatchSize.eq(32).all()),
    "Frozen MRI augmentation reproduced exactly": data_audit["training_image_bitwise_equal"],
    "Existing protected files unchanged": verify_protected(ROOT) == data_audit["protected_files"],
    "Full training remains disabled": training["enabled"] is False,
}
assert all(checks.values()), "Baseline setup audit failed"
write_json(OUTPUT / "baseline_final_audit.json", checks)
display(pd.DataFrame({"Check": checks.keys(), "Status": ["PASS"] * len(checks)}))

## Conclusions and Phase 9 boundary

All conclusions below describe architecture readiness and measured resource requirements. There are no baseline training histories, validation scores, or held-out performance claims. No scientific conclusion about predictive superiority can be drawn from parameter or MAC counts alone.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
passed = int(memory.Status.eq("PASS").sum())
issues = "No batch-32 CUDA OOM was observed." if passed == 5 else "One or more models require a batch-size fairness decision before Phase 9."
display(Markdown(
    f"**Five baseline architectures are prepared.** All use one-channel MRI, three logits, "
    f"and explicit random initialization. Batch-32 forward/loss/backward passed for **{passed}/5** models. "
    f"{issues} The measured memory excludes optimizer-step overhead.\n\n"
    "Frozen folds, preprocessing, main checkpoints, and prior results are preserved. "
    "The approved internal-mask augmentation policy preserves the same MRI transformations while "
    "withholding auxiliary information from the baseline models and loss.\n\n"
    "**No full baseline training has started. Phase 9 requires separate approval.**"
))